# 01 — Data check

**EXSU 500 · MSK 2024 prostate cancer cohort**

Verify the inputs generated by the data-lead scripts. This notebook does not train a model.

## 1. Load processed data

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "processed"

clinical = pd.read_csv(DATA_DIR / "clinical_manifest.csv")
mutations = pd.read_csv(DATA_DIR / "mutations_long.csv")
cna = pd.read_csv(DATA_DIR / "cna_long.csv")
panels = pd.read_csv(DATA_DIR / "gene_panel_matrix.csv")

for name, frame in {"Clinical": clinical, "Mutations": mutations, "CNA": cna, "Panels": panels}.items():
    print(f"{name}: {frame.shape}")

## 2. Verify cohort integrity and class distribution

In [ ]:
assert clinical["PATIENT_ID"].is_unique
assert clinical["SAMPLE_ID"].is_unique
assert clinical["stage_binary"].isin([0, 1]).all()
assert mutations["SAMPLE_ID"].isin(clinical["SAMPLE_ID"]).all()
assert cna["SAMPLE_ID"].isin(clinical["SAMPLE_ID"]).all()
assert panels["SAMPLE_ID"].isin(clinical["SAMPLE_ID"]).all()
assert not cna.duplicated(["SAMPLE_ID", "HUGO_SYMBOL"]).any()

print("Stage distribution:")
print(clinical["stage_binary"].value_counts().sort_index())
print("\nSequencing panels:")
print(clinical["GENE_PANEL"].value_counts())
print("\nSpecimen type vs stage:")
print(pd.crosstab(clinical["SAMPLE_TYPE"], clinical["stage_binary"]))
print("\nPASS: Structural consistency checks")

## 3. Interpretation caveats

- The label is the **highest recorded stage**, not necessarily stage at specimen collection.
- The absence of a mutation record is not proof of wild type unless the gene was covered by the panel.
- In the provided cohort, some patients labelled Stage I–III have metastatic specimens.
- Biological validity and external generalizability are not established by these checks.